# Day 35 · Backprop by hand: the chain rule in action

**First-principles question:** How much does each input change the final answer?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import importlib
import math
import lab.micrograd as micrograd
importlib.reload(micrograd)          # picks up your latest edits to lab/micrograd.py
from lab.micrograd import Value, draw

a = Value(2.0, label="a"); b = Value(-3.0, label="b"); c = Value(10.0, label="c"); f = Value(-2.0, label="f")
e = a * b; e.label = "e"
d = e + c; d.label = "d"
L = d * f; L.label = "L"

## 1 · Gradients by hand
Start at L (its gradient is 1) and walk backwards. Multiply local × upstream.

In [ ]:
# TODO: set every .grad yourself (numbers only, no backward())
L.grad = 1.0
f.grad = None     # dL/df: L = d * f, so the local derivative is d
d.grad = None     # dL/dd
e.grad = None     # dL/de: d = e + c, so local = 1, times upstream d.grad
c.grad = None
a.grad = None     # dL/da: e = a * b, so local = b, times upstream e.grad
b.grad = None
print(draw(L))

## 2 · Check every gradient by nudging (project 3)

In [ ]:
def L_from(av, bv, cv, fv):
    return (av * bv + cv) * fv

H = 1e-6
base = L_from(2.0, -3.0, 10.0, -2.0)
nudged = {
    "a": (L_from(2.0 + H, -3.0, 10.0, -2.0) - base) / H,
    "b": (L_from(2.0, -3.0 + H, 10.0, -2.0) - base) / H,
    "c": (L_from(2.0, -3.0, 10.0 + H, -2.0) - base) / H,
    "f": (L_from(2.0, -3.0, 10.0, -2.0 + H) - base) / H,
}
for name, node in {"a": a, "b": b, "c": c, "f": f}.items():
    print(f"{name}: by hand {node.grad} · nudged {nudged[name]:.4f}")

**What does the gradient of `a` tell you, in one sentence?** …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
for name, node in {"a": a, "b": b, "c": c, "f": f}.items():
    assert node.grad is not None and abs(node.grad - nudged[name]) < 1e-3, f"{name}.grad does not match its nudge"
assert d.grad == -2.0 and e.grad == -2.0, "d and e"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")